# Data preparation

- **Load:** open the file.
- **Clean:** remove extra spaces.
- **Code:** change answers to numbers.
- **Rank and scale:** prepare different versions of the scores.

Data preparation and SQ2 are included. SQ1 and SQ3 are not yet tested. Prepared with AI assistance.

## 1. Load

In [ ]:
import pandas as pd
from pathlib import Path

# Load Excel and make a copy.
df_raw = pd.read_excel(Path.home() / "Downloads" / "Responses (3).xlsx")
df = df_raw.copy()
print(df.shape)

## 2. Clean
Missing answers stay missing. No rows are removed.

In [ ]:
# Remove extra spaces from names and answers.
df.columns = df.columns.str.replace("\u00a0", " ", regex=False).str.strip()
for column in df.select_dtypes(include=["object", "string"]):
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)

## 3. Code
1 = Strongly disagree, 5 = Strongly agree. Each score uses one question.

In [ ]:
# Change the answers to numbers.
scale = {"Strongly disagree": 1, "Disagree": 2, "Neutral": 3,
         "Agree": 4, "Strongly agree": 5}

# Use short names for the three questions.
questions = {
    "visibility": "I feel like my content gets the same number of views as content from others",
    "control": "I feel I am in control of my feed",
    "fairness": "I trust the algorithm to recommend me content fairly"
}

for name, question in questions.items():
    assert df[question].dropna().isin(scale).all()
    df[name] = df[question].map(scale).astype("Float64")

df["age_group"] = df["What age group are you in?"]
scores = df[list(questions)]
scores.head()

## 4. Rank and scale
Ranks use average positions for equal scores. Min-max gives values from 0 to 1. Z-scores show distance from the mean. These are separate options, calculated across all respondents. Missing answers stay missing. Scaling does not make the answers normally distributed.

In [ ]:
# Rank the scores. Equal scores get the same rank.
ranks = scores.rank().add_suffix("_rank")

# Min-max: the lowest value is 0 and the highest is 1.
spread = (scores.max() - scores.min()).replace(0, float("nan"))
minmax = ((scores - scores.min()) / spread).add_suffix("_minmax")

# Z-scores: mean 0 and standard deviation 1.
std = scores.std(ddof=0).replace(0, float("nan"))
z_scores = ((scores - scores.mean()) / std).add_suffix("_z")

# Show the prepared columns without names or emails.
df_prepared = pd.concat([df[["age_group"]], scores, ranks, minmax, z_scores], axis=1)
df_prepared.head().round(2)

## 5. Check the data

In [ ]:
# Check missing answers, age groups and duplicate rows.
print("Missing answers:")
print(scores.isna().sum())
print("Age groups:")
print(df["age_group"].value_counts(dropna=False))
print("Duplicate rows:", df_raw.duplicated().sum())

## 6. Scores by age group

Each chart shows the mean score (1–5). Using a mean treats the gaps between 1, 2, 3, 4 and 5 as equal. Small age groups are less reliable. The charts alone do not show whether the differences are statistically significant.

In [ ]:
import matplotlib.pyplot as plt

# Put age groups in order, from young to old.
age_order = ["Under 18", "18-24", "25-34", "35-44", "45-54", "55-64", "65 and older"]
means = df.groupby("age_group")[list(questions)].mean().reindex(age_order)

# Make one simple chart for each topic.
for column in questions:
    means[column].dropna().plot.bar(color="steelblue", figsize=(7, 4))
    plt.title(column.replace("_", " ").title())
    plt.xlabel("Age group")
    plt.ylabel("Mean score (1–5)")
    plt.ylim(0, 5)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. SQ2 — Perceived control

**Question:** How does perceived control over social media content differ across age groups?

We use “I feel I am in control of my feed”, scored from 1 (Strongly disagree) to 5 (Strongly agree). This question gives one measure of perceived control.

**H0:** The score distributions are the same across age groups.  
**H1:** At least one age group has a different score distribution.

We use the Kruskal-Wallis test because the answers are ordered categories and there are more than two independent groups. We assume each respondent answered once and responses are independent. Missing age or control answers are excluded only from this test.

**Method:** The rank test is the primary analysis. A secondary multiple regression below checks whether age is still associated with control when social media use frequency is included. This frequency may also be associated with perceived control, so the adjusted and unadjusted comparisons answer different questions. Neither analysis estimates a causal effect.


In [ ]:
# Select the data for SQ2.
sq2 = df[['age_group', 'control']].dropna()

# Make a small table to compare the groups.
summary = sq2.groupby('age_group')['control'].agg(['count', 'mean', 'median'])
display(summary.reindex(age_order).dropna(how='all').round(2))
print('Included:', len(sq2), '| Excluded:', len(df) - len(sq2))

In [ ]:
from scipy.stats import kruskal

# Collect the scores for each age group.
groups = []
for age in sq2['age_group'].unique():
    groups.append(sq2.loc[sq2['age_group'] == age, 'control'].astype(float))

# Test whether the score distributions differ.
h, p = kruskal(*groups)

# Adjust for the three planned tests.
p_corrected = min(p * 3, 1)
print('H:', round(h, 2))
print('p-value:', round(p, 6))
print('Corrected p-value:', round(p_corrected, 6))

### Conclusion

Perceived control scores differ across age groups, **H(5) = 24.75, p = 0.000156**. The result remains significant after a Bonferroni correction for the three planned subquestions (**adjusted p = 0.000467**, below 0.05).

Respondents under 18 and aged 18–24 have median scores of 4. The other observed groups have medians of 3. These are descriptive patterns. The test does not tell us which specific pairs of groups differ significantly.

The age groups have different sizes, and some are small (only 7 respondents aged 65+). There are no respondents aged 55–64. Kruskal-Wallis compares distributions; a difference in medians alone requires similarly shaped distributions. The survey shows a link with age. It does not show that age causes control. The item does not measure every aspect of control.

### Simple explanation for the defence

- **What did you study?** Whether perceived control differs between age groups.
- **How did you measure it?** One statement, scored from 1 (Strongly disagree) to 5 (Strongly agree).
- **Why this test?** The answers have an order, and there are more than two groups.
- **What does the p-value mean?** If the score distributions were the same, a result at least this extreme would be unlikely.
- **What did you find?** There is evidence of a difference between groups. This test does not show which pairs differ.
- **What is the main limit?** Some groups are small, and control is measured with one question.


## 8. SQ2 — Multiple regression

**Adjusted question:** Is age group associated with perceived control after accounting for social media use frequency?

This secondary model uses two explanatory variables: age group and social media use frequency. Both are treated as categories. We do not assume equal gaps between the age groups or use labels.

The outcome is the original control score, from 1 to 5. Linear regression treats the steps as equally spaced. That is an extra approximation for a single Likert item, so the rank test remains the primary analysis. HC3 standard errors allow unequal residual variances. They do not fix a wrong model or dependent responses.

**Adjusted H0:** All age-group coefficients are zero after accounting for use frequency.  
**Adjusted H1:** At least one age-group coefficient is not zero.

The reference groups are 18–24 and Always. Each coefficient shows the score difference from the reference group, after accounting for the other variable. This is an extra check. It uses the same data, so it is not independent evidence.

In [ ]:
import statsmodels.formula.api as smf

# Use rows with all answers needed for this model.
model_data = df[['control', 'age_group', 'I use social media']].dropna().copy()
model_data = model_data.rename(columns={'I use social media': 'social_use'})
model_data['control'] = model_data['control'].astype(float)
model_data['age_group'] = model_data['age_group'].astype(str)
model_data['social_use'] = model_data['social_use'].astype(str)

# Use two predictors. C treats a variable as a category.
model = smf.ols('control ~ C(age_group) + C(social_use)', data=model_data).fit(cov_type='HC3')

# Show score differences and confidence intervals.
results = model.params.to_frame('Difference')
results[['CI low', 'CI high']] = model.conf_int()
display(results.round(3))

# Test all age differences together.
display(model.wald_test_terms(scalar=True).table.round(4))
print('Included:', int(model.nobs), '| Excluded:', len(df) - int(model.nobs))
print('R-squared:', round(model.rsquared, 3))

### Assumptions and checks

- **Independent responses:** we found no identical rows or repeated IDs, but this cannot prove that participants are independent. Recruitment and any clustering must be checked by the group.
- **Equal spacing of the outcome:** this is a modelling choice, not something scaling proves. Keep the ordinal rank test as the main result.
- **Unequal variances:** check residuals below and use HC3 standard errors.
- **No perfect collinearity:** check that the design matrix has full rank.
- **Additive model:** the model assumes age differences do not depend on use frequency. It does not estimate interactions. Small groups limit more detailed modelling.
- **Missing answers:** complete-case analysis can be biased if non-response is related to control or age. No answers are imputed.

The residual plot is a diagnostic, not proof that all assumptions hold. Residuals need not be normally distributed for the large-sample HC3 inference, but small groups still make estimates less reliable.

In [ ]:
import numpy as np
from statsmodels.stats.diagnostic import het_breuschpagan

# Check for model columns that fully overlap.
x = model.model.exog
print('Full rank:', np.linalg.matrix_rank(x) == x.shape[1])

# A small p-value suggests unequal variances.
bp = het_breuschpagan(model.resid, x)
print('Breusch-Pagan p-value:', round(bp[1], 4))

# Check whether the spread around zero changes.
plt.scatter(model.fittedvalues, model.resid, alpha=0.25)
plt.axhline(0, color='black')
plt.xlabel('Predicted control score')
plt.ylabel('Residual')
plt.title('SQ2: residual check')
plt.tight_layout()
plt.show()

### Adjusted conclusion

In the multiple regression, age group is still associated with control after accounting for use frequency (joint HC3 Wald test: **χ²(5) = 19.49, p = 0.001554**). The model includes **673 respondents** and excludes 12 with missing required answers. It explains approximately **4.3%** of the variation in control scores. The test finds an age difference, but the model explains only a small part of the differences between people.

This is consistent with the primary rank-test result. The two methods look at the data in different ways and have different assumptions. We do not claim that every age-group pair differs, that age causes control, or that use frequency accounts for every relevant difference between respondents.

### Simple explanation for the defence

- **Why add a second model?** To include age and social media use at the same time.
- **What does C(...) mean?** Treat the variable as categories.
- **What does HC3 do?** Adjust the standard errors for unequal variances.
- **What does 4.3% mean?** The model explains only a small part of the differences in control.
- **What is the main limit?** The model treats the five answer steps as equally spaced. The rank test stays our main analysis.

## Before submission

**Current sources checked:** Brightspace Final assignment and Module guide Business Statistics 26–27, pages 7–8, checked 28 October 2026.

- Deadline: **4 November 2026, 23:59 (Europe/Amsterdam)**.
- This notebook covers the current SQ2 work, not the complete group assignment. Complete SQ1 and SQ3 and the overall methodology and conclusions.
- The assignment page asks for individual findings in a notebook; the module guide asks for one integrated notebook. **Confirm the submission format with the lecturer.**
- Each member needs a unique hypothesis. Confirm that SQ2 is assigned to you within your group.
- Identify the person responsible for each code section/line and credit adapted sources. The existing code was generated with Codex assistance; do not label it as independently written. Add your actual modifications and understanding honestly.
- The theoretical foundation/defence is **50%** of the assignment rubric. Be ready to explain the variables, hypotheses, missing-data choices, tests and limitations.

**Source links:** [Final assignment](https://dlo.mijnhva.nl/d2l/lms/dropbox/user/folder_submit_files.d2l?db=363993&grpid=0&isprv=0&bp=0&ou=774735), [Module guide](https://dlo.mijnhva.nl/d2l/le/lessons/774735/topics/3113282).
